# Lab 2: Feature Extraction and Sleep Spindle Decoding

Build an end-to-end detector: raw EEG → features → decoder score → predictions → held-out evaluation.

## Learning goals

- Extract time- and frequency-domain EEG features.
- Explain filtering delay and the time-frequency tradeoff.
- Compare fixed and causal adaptive thresholds.
- Recognize artifact-contaminated data and apply an artifact-rejection rule.
- Train and evaluate a binary decoder without test-set leakage.

## 1. Setup

Lab 2 uses **SciPy** and **scikit-learn** in addition to the packages installed for Lab 1. Before running this notebook, open a terminal, make sure you are in the repository containing all the lab assignments, activate the conda environment, and install the additional packages:

```bash
conda activate ee194-21 # activate the conda env
```
```bash
python -m pip install scipy scikit-learn # additional packages needed for this lab

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

## 2. Simulated sleep EEG

Sleep spindles are brief bursts of rhythmic neural activity that appear prominently during non-rapid eye movement (NREM) sleep. They are visible in EEG as waxing-and-waning oscillations: their amplitude grows, persists briefly, and then decreases. Spindles are associated with communication between the thalamus and cortex and have been studied in relation to sleep-dependent memory consolidation.

For this lab, we will define a spindle as a 10–13 Hz burst lasting approximately 0.5–1.5 seconds. This is a simplified operational definition for our detector. It is distinct from the much slower, less-than-1-Hz activity commonly called the sleep slow oscillation.

**Optional reading:** [Fernandez and Lüthi (2020), *Sleep Spindles: Mechanisms and Functions*](https://doi.org/10.1152/physrev.00042.2018). This review provides broader biological context for spindle characteristics, thalamocortical mechanisms, temporal organization, and proposed functions.

Real sleep EEG does not come with perfect event labels. Here we simulate the recording so that we know the true start and stop time of each spindle. These labels will later let us determine when a detector succeeds or fails.


The simulated signal contains:

- **$1/f$-like background EEG:** lower frequencies generally contain more power than higher frequencies.
- **A 0.8 Hz slow wave:** a sleep-like oscillation well below the spindle band.
- **Random measurement noise:** broadband fluctuations that make individual events harder to see.
- **Known spindle bursts:** brief 10–13 Hz events with smoothly increasing and decreasing amplitude.
- **Gradual amplitude decay:** signal amplitude decreases across the recording, representing nonstationarity.
- **Two large artifacts near 152 s and 207 s:** non-neural transients with energy that leaks into the spindle band. The first is an extended movement artifact; the second is shorter.

The decay and artifacts are deliberately exaggerated so that we can see how a detector calibrated at the beginning of a recording can fail later. In particular, the first artifact occupies enough of the adaptive detector's recent history to distort its threshold for a real spindle immediately afterward.

In [ ]:
from src.eeg_simulation import simulate_sleep_eeg

fs,time_s,eeg_uv,spindle_mask,spindle_events,gain_drift,artifact_intervals = simulate_sleep_eeg(fs=100, duration_s=240)
print(f"Duration: {time_s[-1]:.1f} s; sampling rate: {fs} Hz; events: {len(spindle_events)}")

### 2.1 Inspect and split the data

The first 60 seconds are calibration data. We will use only this interval to select fixed parameters. Everything afterward is held-out test data that represents future deployment.

First, create Boolean masks named `calibration_mask` and `test_mask` that divide the samples at `calibration_stop_s`. Then plot `eeg_uv` against `time_s` in the upper panel.

The upper plot shows the complete raw recording, including the calibration boundary and artifact intervals. The lower plot zooms in on one spindle. The shaded spindle label is ground truth supplied by the simulator; a deployed detector would have to infer it from the EEG.

In [ ]:
# TODO: Split the recording into calibration and test samples, then plot the complete raw EEG.
calibration_stop_s = 60
calibration_mask = ...  ## student code: mark samples before the calibration boundary
test_mask = ...  ## student code: mark all remaining samples as held-out test data

# Select one known spindle to show at a more useful time scale.
example_start_s,example_stop_s = spindle_events[8]
view = (time_s>=example_start_s-2)&(time_s<=example_stop_s+2)

# Plot the complete recording above and one example event below.
fig,axs = plt.subplots(2,1,figsize=(11,6),dpi=120)
axs[0].plot(...)  ## student code: plot raw EEG voltage against time
axs[0].axvline(calibration_stop_s,color="k",linestyle="--",label="test begins")
for i,(artifact_start,artifact_stop) in enumerate(artifact_intervals):
    axs[0].axvspan(artifact_start,artifact_stop,color="C1",alpha=0.25,label="artifact" if i==0 else None)
axs[0].set(xlabel="Time (s)",ylabel="Voltage (µV)",title="Simulated sleep EEG"); axs[0].legend()
axs[1].plot(time_s[view],eeg_uv[view]); axs[1].axvspan(example_start_s,example_stop_s,color="C3",alpha=0.2,label="true spindle")
axs[1].set(xlabel="Time (s)",ylabel="Voltage (µV)",title="Example spindle"); axs[1].legend(); fig.tight_layout()

print("Calibration/test spindles:",np.sum(spindle_events[:,0]<60),np.sum(spindle_events[:,0]>=60))

**Question 1:** Compare the complete recording with the zoomed example. Even though the shaded interval tells you where the example spindle occurs, why would it be difficult to detect spindles reliably by applying a threshold directly to the raw voltage? Identify two properties that a more useful spindle feature should emphasize.

**Answer:** 

**Question 2:** Later, we will calculate a spindle-detection threshold from the first 60 seconds and evaluate it on the remainder of the recording. Why would calculating that threshold from the complete recording, including the test period, produce an overly optimistic estimate of performance?

**Answer:**

## 3. Filtering and a time-domain feature

Raw EEG contains activity across many frequencies, but our operational spindle definition focuses on 10–13 Hz. We will therefore design a **fourth-order** Butterworth band-pass filter (using `signal.butter`) with cutoffs at 10 and 13 Hz. Use `fs=fs` so SciPy interprets the cutoffs in hertz, and request second-order-section output with `output="sos"` for numerical stability.

We will apply the same filter in two ways. Pass the filter coefficients and raw EEG to `signal.sosfiltfilt` to produce an offline zero-phase result, and to `signal.sosfilt` to produce a causal result. `signal.sosfiltfilt` filters forward and backward and therefore uses future samples. `signal.sosfilt` makes one forward pass, but introduces phase shift and delay.

Finally, summarize the causal filtered signal with a trailing 0.25-second RMS amplitude feature. At each sample, select the window ending at that sample, square its values, average them, and take the square root. For the first samples of the recording, use all samples available so far rather than looking into the future.

In [ ]:
# TODO: Design and apply the band-pass filter, then calculate a trailing RMS feature.
sos = signal.butter(...)  ## student code: design a fourth-order 10–13 Hz Butterworth band-pass filter in SOS form
filtered_zero_phase = signal.sosfiltfilt(...)  ## student code: apply the SOS filter forward and backward for zero-phase filtering
filtered_causal = signal.sosfilt(...)  ## student code: apply the SOS filter once for causal filtering

# Calculate RMS using only the samples available at each point in time.
def trailing_rms(x,window_samples):
    rms_score = np.empty(x.size,dtype=float)
    for end in range(x.size):
        start = max(0,end-window_samples+1)
        window = x[...]  ## student code: select the trailing window ending at the current sample
        rms_score[end] = ...  ## student code: calculate the root-mean-square value of the window
    return rms_score

spindle_score = trailing_rms(...)  ## student code: calculate a 0.25-second trailing RMS of the causal filtered EEG

# Compare both filtered signals and the resulting causal feature.
fig,axs = plt.subplots(2,1,figsize=(11,5),sharex=True,dpi=120)
axs[0].plot(time_s[view],filtered_zero_phase[view],label="zero phase")
axs[0].plot(time_s[view],filtered_causal[view],label="causal",alpha=0.8); axs[0].legend(); axs[0].set_ylabel("Filtered EEG (µV)")
axs[1].plot(time_s[view],spindle_score[view],color="C1"); axs[1].set(xlabel="Time (s)",ylabel="10–13 Hz RMS (µV)")
for ax in axs: ax.axvspan(example_start_s,example_stop_s,color="C3",alpha=0.15)
fig.tight_layout()

**Question 3:** Imagine that this detector must trigger an intervention while EEG is being recorded (online actuation). Which filter must be used? Compare the timing advantage or disadvantage in both filtering methods.

**Answer:** 

**Question 4:** The 10–13 Hz band-pass filter is deliberately narrow. Suppose an online detector needs to respond sooner. How would lowering the filter order or widening its transition band generally affect temporal delay and frequency selectivity? Describe the tradeoff rather than assuming one filter is always best. *(Feel free to try different filter orders in your code above to explore this question. Remember to restore the fourth-order filter before moving on!)*

**Answer:** 

**Question 5:** The trailing RMS currently summarizes the previous 0.25 seconds. If the window were increased to 1 second, how would the stability of the feature change? What would happen to detection latency and to the estimated onset and offset of a short spindle? *(Feel free to try different window sizes in your code above to explore this question. Remember to restore the 0.25-second window before moving on!)*

**Answer:** 

## 4. Fixed and causal adaptive thresholds

The trailing RMS is a continuous feature: larger values indicate stronger recent 10–13 Hz activity. To turn it into a binary detector, we need a rule such as `predict spindle if score ≥ threshold`.

A useful threshold should describe the typical background level plus an allowance for ordinary background variability. We use the **median** for the typical level and the **median absolute deviation (MAD)** for variability:

$$\operatorname{MAD}(x)=\operatorname{median}(|x-\operatorname{median}(x)|).$$

MAD is sometimes used as a more robust measure for spread since a small number of very large values has less influence on it than on standard deviation. This is useful for EEG, where calibration data may contain spindles or artifacts. Our threshold is `median + k × MAD`, with the empirical multiplier `k=6`.

The **fixed threshold** is calculated once from the first 60 seconds and then held constant. Select the calibration RMS samples with `calibration_mask`, calculate their median-plus-MAD threshold using the provided $k=6$, and divide the RMS feature by this threshold to obtain `fixed_score`. A normalized score of 1 is therefore the detection boundary.

The **adaptive threshold** is updated once per second using the *preceding 30 seconds only*. At each update, select the history ending immediately before the current sample, calculate its median-plus-MAD threshold, and assign that value to the next second of `adaptive_threshold`. Then divide the RMS feature by the time-varying threshold to obtain `adaptive_score`. The initial threshold values are set to the fixed calibration threshold until a complete 30-second history is available.

In [ ]:
# TODO: Implement MAD, then calculate fixed and causal adaptive detection thresholds.
def mad(x):
    return ...  ## student code: calculate the median absolute deviation from the median of x

# Calculate one recording-specific threshold from the calibration period.
k = 6.0
calibration_scores = spindle_score[...]  ## student code: select RMS feature samples from the calibration period only
fixed_threshold = ...  ## student code: calculate the calibration median-plus-MAD threshold
fixed_score = ... ## student code: normalize the RMS feature so that 1 is the fixed decision boundary

# Recalculate a causal threshold from recent history once per second.
history_samples,update_samples = 30*fs,fs
adaptive_threshold = np.full(spindle_score.shape,fixed_threshold,dtype=float)
for start in range(history_samples,spindle_score.size,update_samples):
    history = spindle_score[...]  ## student code: select only the preceding 30 seconds of RMS history
    adaptive_threshold[start:start+update_samples] = ...  ## student code: assign the history-based threshold to the next one-second update interval
adaptive_score = ... ## student code: normalize the RMS feature by its time-varying threshold

# Show the same features and thresholds at full and zoomed y-axis scales.
fig,axs = plt.subplots(2,1,figsize=(11,7),sharex=True,dpi=120)
for ax in axs:
    ax.plot(time_s,spindle_score,linewidth=0.7,label="spindle feature (trailing RMS)")
    ax.axhline(fixed_threshold,color="C3",label="fixed threshold")
    ax.plot(time_s,adaptive_threshold,color="C2",label="adaptive threshold")
    ax.axvline(calibration_stop_s,color="k",linestyle="--",label="test begins")
    for i,(artifact_start,artifact_stop) in enumerate(artifact_intervals):
        ax.axvspan(artifact_start,artifact_stop,color="C1",alpha=0.18,label="artifact" if i==0 else None)
    ax.set_ylabel("10–13 Hz RMS (µV)")

axs[0].set_title("Full feature scale")
axs[0].legend(ncol=2)

non_artifact_mask = np.ones(time_s.size,dtype=bool)
for artifact_start,artifact_stop in artifact_intervals:
    non_artifact_mask &= ~((time_s>=artifact_start)&(time_s<=artifact_stop))
zoom_ymax = 1.15*max(
    fixed_threshold,
    np.max(adaptive_threshold),
    np.percentile(spindle_score[non_artifact_mask],99),
)
axs[1].set_ylim(0,zoom_ymax)
axs[1].set_title("Zoomed scale")
axs[1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

**Question 6:** Consider the **fixed threshold** in the lower plot. As the simulated signal amplitude decreases, how do the later spindle-feature peaks change relative to the fixed threshold? What type of detection error can result, and which metric (accuracy, precision, specificity, or sensitivity) most directly reflects that error? Give one plausible recording-related or biological reason that EEG amplitude could change over time.

**Answer:** 

**Question 7:** Now consider the **adaptive threshold** in the lower plot.

**a)** Why does it generally decrease as the recording progresses?

**b)** Inspect the first shaded artifact and the next three aparent true spindles whose onsets occur after the artifact ends, around 158–176 s. Does the artifact cross either threshold? For each of the three following spindles, determine whether its feature peak crosses the fixed threshold, the adaptive threshold, both, or neither. Where do the detectors disagree? *(Feel free to adjust the plot’s `xlim` and `ylim` parameters to inspect these events more closely. Remember to change them back before moving on!)*

**c)** Explain how the adaptive detector's 30-second history produces this disagreement. Suggest one modification that could preserve adaptation to gradual drift while reducing the artifact's effect.

**Answer:**

**a)** 

**b)** 

**c)** 

## 5. Time-frequency analysis

A spectrogram divides a signal into overlapping time windows and calculates a spectrum in each window. Stacking those spectra shows how power at each frequency changes over time. 

Recall that for a window containing $N$ samples at sampling rate $f_s$, the discrete Fourier transform evaluates frequencies separated by

$$\Delta f = \frac{f_s}{N} = \frac{1}{T},$$

where $T=N/f_s$ is the window duration. A longer window therefore gives higher resolution frequency bins, but it combines information over a longer span of time.

### 5.1 Inspect the complete spectrogram

We will first calculate a spectrogram over the complete recording using a 1-second Hann window. Convert the window duration to samples using `fs`, then call `signal.spectrogram` with `eeg_uv`, `fs=fs`, `window="hann"`, the calculated `nperseg`, 75% overlap, and `scaling="density"`.

Consecutive spectrogram windows can overlap, meaning that they share some of the same samples. Here, 75% overlap means that each new window begins after 25% of the previous window has passed. This gives us more closely spaced estimates over time and makes the spectrogram look smoother, but it does not change the frequency spacing set by the window length. For example: with a 1-second window and 75% overlap, the window advances by 0.25 seconds each time. This provides a new spectral estimate every 0.25 seconds instead of every second.

For visualization, retain frequencies from 5 through 20 Hz. Convert the selected power spectral density values to decibels using $10\log_{10}(\mathrm{PSD})$. Add the small constant $10^{-12}$ before taking the logarithm to avoid $\log_{10}(0)$.

In [ ]:
# TODO: Calculate and display a complete-recording spectrogram.
full_window_s = 1.0
full_nperseg = ...  ## student code: convert the one-second window duration to a number of samples
f_full,t_full,sxx_full = signal.spectrogram(...)  ## student code: compute a PSD spectrogram with a Hann window and 75% overlap
full_keep = ...  ## student code: select frequencies from 5 through 20 Hz
full_power_db = ...  ## student code: convert the selected PSD values to decibels

# Display time on the x-axis, frequency on the y-axis, and power as color.
fig,ax = plt.subplots(figsize=(11,4),dpi=120)
mesh = ax.pcolormesh(t_full,f_full[full_keep],full_power_db,shading="auto",cmap="magma")
ax.axvline(calibration_stop_s,color="cyan",linestyle="--",label="test begins")
ax.set(xlabel="Time (s)",ylabel="Frequency (Hz)",title="Complete-recording spectrogram with automatic color limits")
ax.legend()
fig.colorbar(mesh,ax=ax,label="PSD (dB/Hz)")
fig.tight_layout()

**Question 8:** Inspect the bright regions near 152 s and 207 s in the complete spectrogram. How does their frequency extent differ from the concentrated 10–13 Hz activity expected for a spindle? Why do these artifacts make the remaining spindles difficult to see? Describe one plotting change that would make ordinary spindle activity more visible and one data-analysis step that could prevent these artifact-contaminated periods from being treated as valid neural data. Would applying a 10-13 Hz band-pass filter resolve the problem? Why or why not? 

**Answer:** 

### 5.2 Window duration and the time-frequency tradeoff

We will now compare 0.5-second and 2-second Hann windows over the same artifact-free interval (68–93 s). For each provided window duration, convert seconds to samples and calculate a spectrogram of `eeg_zoom` using the same settings as Section 5.1: `fs=fs`, a Hann window, 75% overlap, and `scaling="density"`.

The remaining code selects 5–20 Hz, converts PSD to decibels, and plots both results with shared color limits so that the same color represents the same power. The horizontal cyan bars mark known spindle intervals supplied by the simulated data; they are not inputs to the spectrogram and would not be present in real online data.

In [ ]:
# TODO: Calculate spectrograms using two different window durations.
spectrogram_start_s,spectrogram_stop_s = 68,93
spectrogram_view = (time_s>=spectrogram_start_s)&(time_s<spectrogram_stop_s)
eeg_zoom = eeg_uv[spectrogram_view]
spectrogram_results = []

# Repeat the same analysis with a short and a long window.
for window_s in (0.5,2.0):
    nperseg = ...  ## student code: convert the current window duration from seconds to samples
    f,t_relative,sxx = signal.spectrogram(...)  ## student code: compute the spectrogram using the current window length and 75% overlap
    keep = (f>=5)&(f<=20)
    power_db = 10*np.log10(sxx[keep]+1e-12)
    spectrogram_results.append((window_s,nperseg,f[keep],t_relative+spectrogram_start_s,power_db))

# Use shared robust color limits so the two panels are comparable.
all_zoom_power_db = np.concatenate([result[-1].ravel() for result in spectrogram_results])
color_min,color_max = np.percentile(all_zoom_power_db,(2,98))

# Plot both spectrograms and overlay the simulated ground-truth events.
fig,axs = plt.subplots(2,1,figsize=(11,7),sharex=True,sharey=True,dpi=120)
for ax,(window_s,nperseg,f_zoom,t_zoom,power_db) in zip(axs,spectrogram_results):
    mesh = ax.pcolormesh(t_zoom,f_zoom,power_db,shading="auto",cmap="magma",vmin=color_min,vmax=color_max)
    for event_start,event_stop in spindle_events:
        if event_stop>=spectrogram_start_s and event_start<=spectrogram_stop_s:
            ax.hlines(19.5,max(event_start,spectrogram_start_s),min(event_stop,spectrogram_stop_s),color="cyan",linewidth=4)
    ax.set(ylabel="Frequency (Hz)",title=f"{window_s:.1f}-s window")
    fig.colorbar(mesh,ax=ax,label="PSD (dB/Hz)")
axs[-1].set_xlabel("Time (s)")
fig.tight_layout()

**Question 9:** 

**a)** Recall that $f_s=100$ Hz. For each window duration, $T=0.5$ s and $T=2.0$ s, calculate the number of samples $N$ and the Fourier-bin spacing $\Delta f$. 

**b)** Use the cyan ground-truth intervals to compare the plots. Which window localizes spindle onset and offset more sharply, and which shows frequency more precisely? Explain why improving one type of resolution makes the other worse.

**Answer:** 

**a)** 

**b)** 

## 6. Artifact rejection, feature selection, and logistic regression

The clean spectrogram window made the time-frequency tradeoff easier to see, but the complete recording still contains the two large artifacts. Before fitting a decoder, we will divide the signal into epochs, flag epochs whose signal quality is questionable, select a small set of neural features, and fit logistic regression using calibration data only.

### 6.1 Flag extreme artifacts

The complete spectrogram showed that extreme artifacts can resemble very strong spindle activity. First, divide the recording into non-overlapping 1-second **epochs**. Each epoch will receive one quality-control decision (artifact or not), one feature vector, and one binary spindle label.

Convert the 1-second epoch duration to samples using `fs`. For each epoch, count the `True` samples in `spindle_mask` and label the epoch positive when that count is at least `minimum_spindle_samples = int(0.1*fs)`. This requires at least 0.1 seconds of spindle activity and avoids labels caused by negligible overlap at an epoch boundary.

For quality control, calculate the raw peak-to-peak voltage in each epoch with `np.ptp`. Select the calibration values using `calibration_epoch_mask`, then estimate the cutoff using the same robust form as before, $\mathrm{median} + k\,\mathrm{MAD}$, with the provided `artifact_k=12`. This empirical value is intentionally conservative for this simulated dataset: movement artifacts are much larger than ordinary EEG, and we want to reject only extreme contamination. Mark every epoch whose peak-to-peak voltage exceeds the cutoff as **invalid**, meaning that the decoding system abstains instead of interpreting it as either neural class.

In [ ]:
# TODO: Create epoch labels and flag epochs containing extreme artifacts.
epoch_duration_s = 1.0
epoch_samples = ...  ## student code: convert the one-second epoch duration to samples
epoch_starts = np.arange(0,eeg_uv.size-epoch_samples+1,epoch_samples)
epoch_time_s = (epoch_starts+epoch_samples/2)/fs

# Assign one neural label and one artifact-quality feature to each epoch.
minimum_spindle_samples = int(0.1*fs)
epoch_labels = np.empty(epoch_starts.size,dtype=int)
raw_peak_to_peak = np.empty(epoch_starts.size,dtype=float)
for epoch_index,start in enumerate(epoch_starts):
    stop = start+epoch_samples
    epoch_labels[epoch_index] = ...  ## student code: label whether the epoch contains at least 0.1 seconds of spindle activity
    raw_peak_to_peak[epoch_index] = ...  ## student code: calculate the epoch's raw peak-to-peak voltage

# Divide complete epochs into calibration and held-out periods.
calibration_epoch_mask = epoch_time_s<calibration_stop_s
test_epoch_mask = ~calibration_epoch_mask

# Estimate the artifact cutoff without using held-out data.
artifact_k = 12.0
calibration_peak_to_peak = ...  ## student code: select peak-to-peak values from calibration epochs only
artifact_threshold = ...  ## student code: calculate the calibration median-plus-MAD artifact cutoff
artifact_epoch_mask = ...  ## student code: flag epochs whose peak-to-peak voltage exceeds the cutoff

fig,ax = plt.subplots(figsize=(11,3.5),dpi=120)
ax.plot(epoch_time_s,raw_peak_to_peak,marker=".",linewidth=0.8,label="epoch peak-to-peak voltage")
ax.axhline(artifact_threshold,color="C3",label="artifact threshold")
ax.axvline(calibration_stop_s,color="k",linestyle="--",label="test begins")
for i,(artifact_start,artifact_stop) in enumerate(artifact_intervals):
    ax.axvspan(artifact_start,artifact_stop,color="C1",alpha=0.18,label="artifact" if i==0 else None)
ax.set(xlabel="Epoch center (s)",ylabel="Peak-to-peak voltage (µV)",title="Artifact-quality feature")
ax.legend(ncol=2)
fig.tight_layout()
print("Flagged epoch centers (s):",epoch_time_s[artifact_epoch_mask])
print("Calibration epochs (no spindle, spindle):",np.bincount(epoch_labels[calibration_epoch_mask],minlength=2))
print("Test epochs (no spindle, spindle):",np.bincount(epoch_labels[test_epoch_mask],minlength=2))

**Question 10:** Why do we mark extreme epochs invalid instead of automatically labeling them as `no spindle`?

**Answer:** 

**Question 11:** Why do we require at least 0.1 seconds of spindle activity instead of labeling an epoch positive after any single spindle sample? How would increasing the requirement to 0.5 seconds affect events that cross epoch boundaries?

**Answer:** 

### 6.2 Feature selection

A threshold detector makes its decision from one feature and a hand-designed boundary. A learned decoder can instead combine several measurements. We will begin with three candidate neural features motivated by the spindle definition and the feature-extraction lecture:

- **Absolute 10–13 Hz power:** the total amount of spindle-band activity.
- **Relative 10–13 Hz power:** spindle-band power divided by total 5–20 Hz power, measuring how concentrated the spectrum is in the spindle band.
- **Maximum trailing RMS:** the strongest causal spindle-band amplitude reached during the epoch.

For each 1-second epoch, calculate a periodogram using `signal.periodogram` with `fs=fs`, a Hann window, and `scaling="density"`. Integrate the PSD over the provided frequency masks with `np.trapezoid`. Calculate relative power as spindle-band power divided by 5–20 Hz power, adding $10^{-12}$ to the denominator for numerical safety. For the RMS feature, take the maximum value of `spindle_score` within the same epoch.

Absolute power and RMS are positive-valued and strongly right-skewed, so the provided code takes their base-10 logarithm to compress extreme values and make their distributions easier to compare. Relative power is already a bounded ratio, so we leave it untransformed. We visualize only valid calibration epochs: feature selection using the held-out test period would leak future information into the model design. After inspecting the plots, combine relative spindle-band power and log maximum RMS—in that order—to form the final two-column `feature_matrix`.

In [ ]:
# TODO: Extract candidate features from each epoch and assemble the selected feature matrix.
spindle_band_power = []
relative_spindle_power = []
epoch_max_rms = []

# Extract the three candidate features from every epoch.
for start in epoch_starts:
    stop = start+epoch_samples
    epoch = eeg_uv[start:stop]
    f_epoch,psd_epoch = signal.periodogram(...)  ## student code: calculate the epoch PSD using a Hann-window periodogram
    spindle_bins = (f_epoch>=10)&(f_epoch<=13)
    broad_bins = (f_epoch>=5)&(f_epoch<=20)
    band_power = np.trapezoid(...)  ## student code: integrate PSD over the 10–13 Hz spindle band
    broad_power = np.trapezoid(psd_epoch[broad_bins],f_epoch[broad_bins])

    spindle_band_power.append(band_power)
    relative_spindle_power.append(...)  ## student code: divide spindle-band power by total 5–20 Hz power
    epoch_max_rms.append(...)  ## student code: save the maximum causal RMS value in the epoch

spindle_band_power = np.asarray(spindle_band_power)
relative_spindle_power = np.asarray(relative_spindle_power)
epoch_max_rms = np.asarray(epoch_max_rms)

# Log-transform the two strongly right-skewed amplitude features.
log_spindle_band_power = np.log10(spindle_band_power+1e-12)
log_epoch_max_rms = np.log10(epoch_max_rms+1e-12)

candidate_features = np.column_stack((log_spindle_band_power,relative_spindle_power,log_epoch_max_rms))
candidate_names = ("Log 10–13 Hz power","Relative 10–13 Hz power","Log maximum RMS")
model_train_mask = calibration_epoch_mask&~artifact_epoch_mask
model_test_mask = test_epoch_mask&~artifact_epoch_mask

# Compare feature distributions using valid calibration epochs only.
fig,axs = plt.subplots(1,3,figsize=(12,3.3),dpi=120)
for ax,values,name in zip(axs,candidate_features.T,candidate_names):
    plotted = values[model_train_mask]
    bins = np.linspace(plotted.min(),plotted.max(),11)
    ax.hist(values[model_train_mask&(epoch_labels==0)],bins=bins,alpha=0.6,label="no spindle")
    ax.hist(values[model_train_mask&(epoch_labels==1)],bins=bins,alpha=0.6,label="spindle")
    ax.set(xlabel=name,ylabel="Calibration epochs")
axs[0].legend(fontsize=8)
fig.suptitle("Candidate feature distributions: valid calibration epochs")
fig.tight_layout()

# Check whether candidate features contain redundant information.
feature_correlation = np.corrcoef(candidate_features[model_train_mask],rowvar=False)
fig,ax = plt.subplots(figsize=(5,4),dpi=120)
image = ax.imshow(feature_correlation,vmin=-1,vmax=1,cmap="coolwarm")
ax.set_xticks(range(3),candidate_names,rotation=35,ha="right")
ax.set_yticks(range(3),candidate_names)
for row in range(3):
    for col in range(3):
        ax.text(col,row,f"{feature_correlation[row,col]:.2f}",ha="center",va="center")
ax.set_title("Calibration feature correlations")
fig.colorbar(image,ax=ax,label="Pearson correlation")
fig.tight_layout()

# Keep the two complementary features selected for model fitting.
feature_matrix = np.column_stack(...)  ## student code: combine relative power and log maximum RMS as two feature columns
feature_names = ("Relative 10–13 Hz power","Log maximum RMS")

**Question 12:**

**a)** Based on the calibration distributions, which features tend to increase during spindle epochs?

**b)** The correlation plot shows that log absolute spindle-band power and relative spindle-band power are highly correlated. Why might keeping both be unnecessary, particularly with a small calibration set? Explain why relative spindle-band power and log maximum RMS form a useful two-feature representation.

**Answer:**

**a)** 

**b)** 

### 6.3 Fit logistic regression

Recall from lecture, logistic regression models the probability that an epoch contains a spindle as

$$P(y_{\mathrm{epoch}}=1\mid\mathbf{x}_{\mathrm{epoch}})=\sigma(\mathbf{w}^{T}\mathbf{x}_{\mathrm{epoch}}+b),$$

$$\sigma(z)=\frac{1}{1+e^{-z}}.$$

Here, $\mathbf{w}$ contains one learned weight per feature and $b$ is the learned intercept. Fitting chooses $\mathbf{w}$ and $b$ by maximum likelihood, equivalently by minimizing binary log loss on the labeled calibration epochs. Because spindle epochs are less common, the provided `class_weight="balanced"` setting gives the two classes equal total influence during fitting instead of allowing the more common `no spindle` epochs to dominate the objective.

Fit the provided `StandardScaler` on the valid calibration rows selected by `model_train_mask` and transform those rows to create `X_train`. Then fit the provided `logistic_decoder` using `X_train` and `y_train`.

In [ ]:
# TODO: Scale the calibration features and fit logistic regression.
scaler = StandardScaler()
X_train = scaler.fit_transform(...)  ## student code: fit the scaler on valid calibration features and transform them
# Apply the calibration-derived scaling to every epoch without refitting.
X_all_scaled = scaler.transform(feature_matrix)
y_train = epoch_labels[model_train_mask]

logistic_decoder = LogisticRegression(class_weight="balanced",random_state=0)
logistic_decoder.fit(...)  ## student code: fit logistic regression using the scaled calibration features and labels
logistic_probability = logistic_decoder.predict_proba(X_all_scaled)[:,1]

# Evaluate the fitted model over a grid to visualize its decision plane.
x_grid = np.linspace(X_train[:,0].min()-0.5,X_train[:,0].max()+0.5,160)
y_grid = np.linspace(X_train[:,1].min()-0.5,X_train[:,1].max()+0.5,160)
xx,yy = np.meshgrid(x_grid,y_grid)
grid_probability = logistic_decoder.predict_proba(np.column_stack((xx.ravel(),yy.ravel())))[:,1].reshape(xx.shape)

# Plot class examples, predicted probabilities, and the 0.5 boundary.
fig,ax = plt.subplots(figsize=(6,4.5),dpi=120)
probability_map = ax.contourf(xx,yy,grid_probability,levels=np.linspace(0,1,11),cmap="RdBu_r",alpha=0.35)
ax.contour(xx,yy,grid_probability,levels=(0.5,),colors="k",linewidths=1.5)
for label,color,name in ((0,"C0","no spindle"),(1,"C3","spindle")):
    selected = y_train==label
    ax.scatter(X_train[selected,0],X_train[selected,1],s=25,color=color,edgecolor="white",linewidth=0.4,label=name)
ax.plot([],[],color="k",linewidth=1.5,label="probability = 0.5")
ax.set(xlabel=f"Standardized {feature_names[0]}",ylabel=f"Standardized {feature_names[1]}",title="Logistic regression: calibration data and 0.5 boundary")
ax.legend(fontsize=8)
fig.colorbar(probability_map,ax=ax,label="Predicted probability of spindle")
fig.tight_layout()

**Question 13:** The black line shows the default probability boundary of 0.5. If the boundary were lowered below 0.5, how would the region classified as `spindle` change? What would generally happen to sensitivity and specificity? What would happen if the boundary were raised instead?

**Answer:** 

## 7. Held-out decoder performance

We now have three decoders: the fixed-threshold and adaptive-threshold rules from Section 4, and logistic regression. The threshold rules apply hand-designed boundaries to one causal RMS feature. Logistic regression learns how to combine the two selected features using valid calibration epochs.

For comparison, every decoder makes one decision per 1-second epoch. Each threshold score is the maximum normalized RMS value in the epoch, with 1 as its decision boundary. Logistic regression produces a probability, with 0.5 as the default decision boundary.

For all three decoders, evaluation includes only held-out epochs that passed artifact rejection. However, the adaptive threshold was calculated earlier from the continuous signal, before artifact rejection was introduced. Epochs flagged by the artifact-rejection rule are excluded when performance is calculated. However, their samples remain in the adaptive detector’s trailing 30-second history and can temporarily raise its threshold (as explored in Section 4). A more robust implementation could detect artifacts first and exclude them from adaptive-threshold updates, but we won't implement that in this lab.

In [ ]:
# Reduce each continuous threshold score to one maximum value per epoch.
fixed_epoch_score = np.asarray([np.max(fixed_score[start:start+epoch_samples]) for start in epoch_starts])
adaptive_epoch_score = np.asarray([np.max(adaptive_score[start:start+epoch_samples]) for start in epoch_starts])

# Store all three continuous scores and their default decision boundaries.
decoder_scores = {
    "Fixed threshold": fixed_epoch_score,
    "Adaptive threshold": adaptive_epoch_score,
    "Logistic regression": logistic_probability,
}
decision_cutoffs = {
    "Fixed threshold": 1.0,
    "Adaptive threshold": 1.0,
    "Logistic regression": 0.5,
}

### 7.1 Accuracy, sensitivity, specificity, and AUROC

For binary spindle labels:

- **Sensitivity** $=TP/(TP+FN)$: fraction of true spindle epochs detected.
- **Specificity** $=TN/(TN+FP)$: fraction of true non-spindle epochs rejected.
- **Accuracy** $=(TP+TN)/N$: fraction of all epochs classified correctly.
- **AUROC**: probability that a classifier assigns a higher score to a randomly selected positive epoch ($y_{\text{epoch}}=1$) than to a randomly selected negative epoch ($y_{\text{epoch}}=0$).

Complete the accuracy, sensitivity, and specificity calculations in `decoder_metrics` using the provided `tn`, `fp`, `fn`, and `tp` counts. The remaining code evaluates the three decoding approaches and constructs their confusion matrices and ROC curves.

In [ ]:
# TODO: Calculate accuracy, sensitivity, and specificity from confusion-matrix counts.
def decoder_metrics(y_true,score,cutoff):
    prediction = score>=cutoff
    tn,fp,fn,tp = confusion_matrix(y_true,prediction,labels=(0,1)).ravel()
    accuracy = ...  ## student code: calculate the accuracy
    sensitivity = ... ## student code: calculate the sensitivity
    specificity = ...  ## student code: calculate the specificity
    auroc = roc_auc_score(y_true,score)
    return accuracy,sensitivity,specificity,auroc,np.asarray([[tn,fp],[fn,tp]])

# Evaluate every approach on the same valid held-out epochs.
y_test = epoch_labels[model_test_mask]
results = {}
print(f"{'Decoder':<25} {'Accuracy':>9} {'Sensitivity':>12} {'Specificity':>12} {'AUROC':>8}")
for name,all_scores in decoder_scores.items():
    results[name] = decoder_metrics(y_test,all_scores[model_test_mask],decision_cutoffs[name])
    accuracy,sensitivity,specificity,auroc,_ = results[name]
    print(f"{name:<25} {accuracy:9.3f} {sensitivity:12.3f} {specificity:12.3f} {auroc:8.3f}")

majority_accuracy = np.mean(y_test==0)
print(f"\nAlways predicting 'no spindle' would have accuracy {majority_accuracy:.3f}.")

In [ ]:
# Visualize the four outcome counts at each approach's default cutoff.
fig,axs = plt.subplots(1,3,figsize=(11,3.5),dpi=120)
for ax,(name,result) in zip(axs,results.items()):
    matrix = result[-1]
    ConfusionMatrixDisplay(matrix,display_labels=("no spindle","spindle")).plot(ax=ax,colorbar=False,cmap="Blues")
    ax.set_title(name)
fig.tight_layout()

In [ ]:
# Trace sensitivity and false-positive rate as each cutoff is varied.
fig,ax = plt.subplots(figsize=(6,5),dpi=120)
for name,all_scores in decoder_scores.items():
    test_scores = all_scores[model_test_mask]
    false_positive_rate,true_positive_rate,_ = roc_curve(y_test,test_scores)
    ax.plot(false_positive_rate,true_positive_rate,label=f"{name} (AUC={results[name][3]:.3f})")
ax.plot((0,1),(0,1),"k--",label="chance")
ax.set(xlabel="False-positive rate (1 − specificity)",ylabel="True-positive rate (sensitivity)",title="Held-out, artifact-free epochs")
ax.set_aspect("equal",adjustable="box")
ax.legend(fontsize=8)
fig.tight_layout()

**Question 14:**

**a)** Accuracy and AUROC look fairly high for all three decoders. Compare each accuracy with the accuracy obtained by always predicting `no spindle`. Why does the always-negative rule achieve decent accuracy? What additional information do sensitivity, specificity, the confusion matrices, and AUROC provide?

**b)** Compare the fixed and adaptive thresholds. Which has greater sensitivity? What change in false negatives produces this difference, and what happens to specificity?

**Answer:**

**a)** 

**b)** 

## 8. Cool spindle detection research 

Online spindle detection can be used to trigger an intervention while a spindle is still occurring. In a recent human study, a real-time EEG detector triggered brief auditory stimulation during detected spindles. The effect depended on stimulation timing: stimulation resulted in increased sigma-band activity about one second poststimulation, but stimulation delivered near spindle onset could shorten the ongoing spindle. This demonstrates why a closed-loop decoder must determine both *whether* an event is present and *when* it is detected, with sufficiently low latency. See [Jourde et al. (2025)](https://pmc.ncbi.nlm.nih.gov/articles/PMC12230793/).

Spindles also have temporal organization beyond a single event. A recent animal study distinguished isolated spindles from **spindle trains**, or groups of nearby spindles, and found that trains were associated with stronger slow-oscillation coupling and more sustained motor memory reactivation during NREM sleep. See [Darevsky et al. (2024)](https://pmc.ncbi.nlm.nih.gov/articles/PMC11502226/). Our current approaches assign a binary spindle label to each 1-second epoch, but they do not explicitly represent relationships between successive spindles.

**Question 15:** 

**a)** Imagine extending this lab into a closed-loop system with three output states: `no spindle`, `isolated spindle`, and `spindle train`. Propose two or three causal history features or state-transition rules that could help identify a spindle train. Because a detected spindle could be followed by another spindle, when can an online system confidently label the first spindle as `isolated`? Describe how a provisional state could handle this uncertainty.

**b)** Explain how false positives and false negatives would affect (1) a system that plays a sound during a spindle and (2) a clinical system that warns of seizure onset. How would those consequences influence whether you prioritize sensitivity or specificity?

**Answer:** 

**a)** 

**b)** 


## Submission checklist

- Run all completed cells from top to bottom and answer the questions.
- Save the notebook.
- Submit the notebook itself to Gradescope.
- Export a PDF version and submit to Gradescope.